# Full run and MLOps

**Goal:** the moment the project was built for.
1. **One** honest evaluation on the untouched test night (2019-02-08), using the planted fakes.
2. The real **search** on that night (no fakes): known asteroids recovered, unknown candidates ranked for vetting.
3. The engineering that makes it trustworthy: provenance, reproducibility, a model card.

Read `docs/full_run_mlops.md` alongside this notebook.

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from sklearn.metrics import average_precision_score

from asthunt.config.load_config import load_config
from asthunt.dataset.make_unit_tag import make_unit_tag
from asthunt.acquire.load_unit_metadata import load_unit_metadata
from asthunt.acquire.query_ztf_metadata import query_ztf_metadata
from asthunt.features.compute_tracklet_features import FEATURE_COLUMNS
from asthunt.models.load_model_bundle import load_model_bundle
from asthunt.evaluate.compute_classification_metrics import compute_classification_metrics
from asthunt.evaluate.measure_completeness import measure_completeness
from asthunt.pipeline.run_night_pipeline import run_night_pipeline

project = load_config("project"); paths = project["paths"]
configs = {name: load_config(name) for name in ["acquire", "inject", "detect", "link"]}
acquire = configs["acquire"]; field = acquire["field"]


def unit_of(night):
    """The original quadrant (configs/acquire.yaml: field, ccd, quadrant) on one night."""
    return {"field": acquire["field"], "ccd": acquire["ccd"], "quadrant": acquire["quadrant"], "night": night}


test_night = acquire["nights"]["test"][0]
device = "mps" if torch.backends.mps.is_available() else "cpu"

# One model bundle = one version of the whole system (CNN + tracklet classifier + thresholds)
bundle = load_model_bundle(paths["models"], "v1")
realbogus_model, tracklet_model, thresholds = bundle["realbogus_model"], bundle["tracklet_model"], bundle["thresholds"]
realbogus_file = tracklet_file = paths["models"] / "v1"
print("test night:", test_night); print("bundle:", bundle["info"]["version"], bundle["info"]["description"]); print("thresholds:", thresholds)

## 1. The one-time test evaluation (with planted fakes)
Every threshold was fixed on validation nights. We look at the test night **once**, report what we get, and don't tune afterwards.

In [ ]:
tag = make_unit_tag(unit_of(test_night))
detections = pd.read_parquet(paths["processed"] / f"detections__{tag}.parquet")
detections["score"] = np.load(paths["processed"] / f"realbogus_scores__{tag}.npy")
m = compute_classification_metrics(detections["label"], detections["score"], thresholds["realbogus"])
print(f"detections: {len(detections)}, real {detections['label'].sum()}")
print(f"real/bogus CNN on test: AP {m['average_precision']:.3f}, recall {m['recall']:.3f} at the linking threshold, "
      f"bogus kept per frame {m['false_positives'] / detections['frame_index'].nunique():.0f}")

tracklets = pd.read_parquet(paths["processed"] / f"tracklets__{tag}.parquet")
tracklets["tracklet_score"] = tracklet_model.predict_proba(np.nan_to_num(tracklets[FEATURE_COLUMNS].to_numpy()))[:, 1]
t = compute_classification_metrics(tracklets["label"], tracklets["tracklet_score"], thresholds["tracklet"])
print(f"tracklet classifier on test: AP {t['average_precision']:.3f}, recall {t['recall']:.3f}, precision {t['precision']:.3f}, "
      f"bogus tracklets kept {t['false_positives']}")

## 2. End-to-end completeness on the test night: planted asteroid → final candidate

In [ ]:
injections = pd.read_parquet(paths["processed"] / f"injections__{tag}.parquet")
per_fake = injections.groupby("injection_id").agg(mag=("mag", "first"), rate=("rate_arcsec_per_hour", "first"))
found = set(tracklets.loc[(tracklets["label"] == 1) & (tracklets["tracklet_score"] >= thresholds["tracklet"]), "injection_id"])
per_fake["found"] = per_fake.index.isin(found)

curve = measure_completeness(per_fake.reset_index(), "found", "mag", np.arange(17, 22.5, 0.5))
plt.errorbar(curve["bin_centre"], curve["completeness"], curve["uncertainty"], fmt="o-")
plt.xlabel("magnitude"); plt.ylabel("fraction of planted asteroids recovered as candidates"); plt.ylim(0, 1)
plt.title(f"test night {test_night}: end-to-end completeness"); plt.show()
curve.round(2)

In [ ]:
half = curve[curve["completeness"] >= 0.5]
print(f"50% completeness reached down to about mag {half['bin_centre'].max():.2f}" if len(half) else "never reaches 50%")

## 3. The search: no fakes, real sky

In [ ]:
metadata = load_unit_metadata(unit_of(None), paths["raw"])
rng = np.random.default_rng([project["seed"], int(test_night.replace("-", ""))])
result = run_night_pipeline(unit_of(test_night), metadata, configs, bundle, paths, rng, device)
found_tracklets, candidates = result["tracklets"], result["candidates"]
known_found = found_tracklets[found_tracklets["known_name"] != ""]
print(f"{len(result['detections'])} detections -> {len(found_tracklets)} tracklets")
print(f"known asteroids recovered: {len(known_found)}")
print(f"unknown candidates for vetting: {len(candidates)}")

In [ ]:
known_found[["known_name", "n_detections", "rate_arcsec_per_hour", "mag_mean", "tracklet_score"]].round(2).sort_values("mag_mean")

### How many of the known asteroids that we *should* see did we recover?

In [ ]:
frames = result["detections"]["exposure"].unique()
known = pd.concat([pd.read_csv(paths["interim"] / "known_asteroids" / f"known_asteroids__{e}.csv", dtype={"name": str}) for e in frames])
expected = known.groupby("name").agg(v_mag=("v_mag", "mean"), frames=("x", "size"))
expected = expected[expected["frames"] >= configs["link"]["min_detections"]]
expected["found"] = expected.index.isin(known_found["known_name"])
by_mag = expected.groupby(pd.cut(expected["v_mag"], [15, 18, 19, 19.5, 20, 20.5, 21, 22, 25]), observed=True)["found"].agg(["size", "sum", "mean"])
by_mag.columns = ["known in field", "recovered", "fraction"]
by_mag.round(2)

### The unknown candidates

In [ ]:
print("report:", paths["reports"] / test_night / "index.html")
print("draft ADES file:", paths["reports"] / test_night / "candidates_draft_ades.psv")
candidates[["tracklet_id", "tracklet_score", "n_detections", "rate_arcsec_per_hour", "mag_mean", "mag_std", "score_min"]].round(3)

**How to read this honestly:** the validation nights told us roughly how many bogus tracklets pass the threshold per night.
If the number of candidates is similar, most are probably bogus. Each one goes through the vetting checklist
(`docs/crossmatch_vetting.md`), and anything that survives needs **a second night** before it means anything.

## 4. MLOps: can someone else get exactly this?

In [ ]:
import subprocess
print("code version:", subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True,
                                      cwd=paths["models"].parent).stdout.strip())
for f in sorted(paths["models"].glob("*")):
    print(f"{f.stat().st_size / 1e3:8.0f} kB  {f.name}")
print("thresholds:", thresholds)

Everything needed to reproduce this run: the git commit (code + configs), `environment.yml`, the metadata cache
(which exposures), the seeds (project seed + night date), the model files (named with date + commit), and `thresholds.json`.
`docs/full_run_mlops.md` lists the commands and the model card.

## Exercises
1. Which magnitude range do the recovered known asteroids span? Compare with the injected completeness curve: do fakes and real asteroids agree?
2. Vet the top three candidates with the checklist. Write down your verdict and reason for each.
3. Pick another night with ≥ 5 frames from the metadata (not in any split) and run the search on it. What would you need to add to `configs/acquire.yaml`?